# LLM 11일차 — 수업 흐름도(RAG.png)대로 `실습8` 완성 : 분류 → (검색) → **하나의 `answer_node`** · 비동기 · 체크포인팅

수업에서 받은 흐름도 [`RAG.png`](./RAG.png) 를 코드로 옮긴 노트입니다. 오늘 받은 필기 `LangGraph_실습6_1` · `LangGraph_실습7` · `LangGraph_실습8` 은 **Day 055 에 올린 파일과 내용이 같아**(바이트 단위로 동일) 정리 노트는 Day 055 폴더를 보면 되고, 여기서는 **흐름도의 설계**를 그대로 구현해 Day 055 정리판(답변 노드 2개)과 비교했습니다.
(교재 `AI Agent` **p.138** — 실습 : 멀티 에이전트 RAG 시스템 업그레이드 · 체크포인팅 기능 구현)

| 흐름도 | 내용 |
|---|---|
| 루프 | 사용자 입력 → `exit` 이면 종료 · SQLite 연결 해제 / 아니면 `await agent.ainvoke(initial_state + thread_id)` → 마지막 AIMessage 출력 → 다시 입력 |
| 1. 상태 복원 | `AsyncSqliteSaver` 가 `thread_id='user-8888'` 로 SQLite 조회 → `add_messages` 리듀서로 과거 대화 + 새 질문 병합 |
| 2. 워크플로우 | `classifier_node`(분류 · **`context=''`**) → `route_by_rag_need` → RAG : `retriever_node`(재구성 · Chroma 검색) → `answer_node` / NO_RAG : 바로 `answer_node` |
| `answer_node` | 하나의 노드 안에서 RAG 면 `rag_qa_chain`, 아니면 `general_qa_chain` |
| 3. 상태 저장 | SQLite 에 AgentState 저장 |

- 필기 원본에서 고친 부분 · 보충은 **📝** 로 적었습니다. **애니메이션(GIF)** 은 [`images/`](./images/) 폴더에 있습니다.

> **출력 읽는 법** — 정리 노트북은 **API 키 없이** 만들었다.
>
> | 표시 | 뜻 |
> |---|---|
> | `# 📼` | **수업 시간에 실제로 실행한 결과**를 필기 원본에서 그대로 옮긴 출력 |
> | `# ⏳` | API 키가 필요해 **출력이 비어 있는 셀** — 직접 실행 |
> | `# 🧪` | OpenAI 대신 **로컬 임베딩** · **가짜 LLM** 으로 실제로 돌린 셀 — 답변 문장은 gpt-5-nano 와 다르다. **체크포인트 저장 · 복원 · thread_id 분리 · 오류는 진짜와 같다** |
> | 표시 없음 | 정리 노트북에서 실제로 실행한 셀 |

## 0. 설치 · 실행 방법

`pip install langgraph langgraph-checkpoint-sqlite aiosqlite langchain-openai langchain-community langchain-chroma python-dotenv` (키 없이 : `+ langchain-huggingface sentence-transformers`) · 같은 폴더에 `faq_manual.csv` · `.env` · 키가 있으면 `# ⏳`, 없으면 `# 🧪` 셀 (정리 환경 : `langgraph 1.2.21` · `langgraph-checkpoint-sqlite 3.1.3`)

In [1]:
import os, asyncio, warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
from typing import Annotated
from dotenv import load_dotenv, find_dotenv
from pydantic import BaseModel
from langchain_community.document_loaders import CSVLoader
from langchain_chroma import Chroma
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from langgraph.graph import StateGraph, END, add_messages
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver

loaded = load_dotenv(find_dotenv(usecwd=True), override=True)          # 키 값은 출력하지 않는다
print('.env 읽음 :', loaded, '| OPENAI_API_KEY 있음 :', bool(os.getenv('OPENAI_API_KEY')))
from typing import Literal

.env 읽음 : False | OPENAI_API_KEY 있음 : False


In [2]:
docs = CSVLoader('faq_manual.csv', encoding='utf-8', content_columns=['question', 'answer'],
                 metadata_columns=['faq_id', 'category']).load()

In [ ]:
# ⏳ API 키가 필요한 셀 — 정리 환경에서는 실행하지 않음 (직접 실행해 보기)

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
llm = ChatOpenAI(model='gpt-5-nano')
model = OpenAIEmbeddings(model='text-embedding-3-small')

In [4]:
# 🧪 OpenAI 임베딩 대신 로컬 한국어 임베딩(jhgan/ko-sroberta-multitask)으로 실제 실행 — 순위 · 거리는 text-embedding-3-small 과 다를 수 있다

from langchain_huggingface import HuggingFaceEmbeddings
model = HuggingFaceEmbeddings(model_name='jhgan/ko-sroberta-multitask', encode_kwargs={'normalize_embeddings': True})

In [5]:
# 🧪 LLM 대신 '정해 둔 규칙으로 답하는' 가짜 LLM — 체크포인터가 무엇을 저장 · 복원하는지 확인용

GREETING = ('안녕', '고마', '감사', '반가')

def _fake(prompt_value):
    msgs = prompt_value.to_messages()
    system, last, history = msgs[0].content, msgs[-1].content, msgs[1:-1]
    if "'RAG' 또는 'NO_RAG'" in system:                       # 의도 분류
        return AIMessage('NO_RAG' if any(w in last for w in GREETING) else 'RAG')
    if '재구성' in system:                                     # 질문 재구성 : 앞 사용자 질문 + 지금 질문 (Day 053 규칙)
        prev = [m.content for m in history if isinstance(m, HumanMessage)]
        return AIMessage(f'{prev[-1]} {last}' if prev else last)
    if 'answer: ' in system:                                   # 매뉴얼 기반 답변
        first = system.split('answer: ')[-1].split('. ')[0].strip()
        return AIMessage(f'(모의 · 기록 {len(history)}개) {first}.')
    return AIMessage(f'(모의 일반 답변 · 기록 {len(history)}개) 네, 무엇이든 물어보세요!')

llm = RunnableLambda(_fake)

In [6]:
vectorstore = Chroma.from_documents(docs, model, collection_name='faq_flowchart')
retriever = vectorstore.as_retriever(search_kwargs={'k': 1})

def chain(system):
    return ChatPromptTemplate.from_messages([('system', system), MessagesPlaceholder('chat_history'), ('human', '{input}')]) | llm

classifier_chain = chain('''당신은 '코드크래프터스 아카데미'의 고객 문의 분류 전문가입니다. 이전 대화와 최신 질문을 보고
매뉴얼 조회가 필요하면(후속 질문 포함) 'RAG', 인사 · 감사 · 일반 상식이면 'NO_RAG'. 반드시 'RAG' 또는 'NO_RAG' 단어 하나만 출력하세요.''') | StrOutputParser()
rephrase_chain = chain('''주어진 채팅 기록과 최신 질문이 있을 때, 맥락을 참조하는 모호한 질문을 기록 없이도 이해할 수 있는
명확한 질문으로 재구성하세요. 답하지 말고 질문만 반환하세요.''') | StrOutputParser()
rag_qa_chain = chain('''당신은 코드크래프터스 아카데미의 친절한 고객 지원 담당자입니다. 아래 [매뉴얼 내용]과 이전 대화로 답하세요.
매뉴얼에 없는 절차나 기능은 약속하지 마세요.

[매뉴얼 내용]
{context}''')
general_qa_chain = chain('당신은 친절하고 명쾌한 AI 비서입니다. 이전 대화 맥락을 기억하며 자연스럽게 답하세요.')

## 1. 상태 · 노드 — 흐름도 그대로

In [7]:
class AgentState(BaseModel):
    messages: Annotated[list[BaseMessage], add_messages] = []
    needs_rag: bool = False
    context: str = ''

def parse_decision(text: str) -> bool:
    return text.strip().strip('\'"`.').strip().upper() == 'RAG'

async def classifier_node(state: AgentState) -> dict:                       # [Node 1]
    history, question = state.messages[:-1], state.messages[-1].content
    decision = await classifier_chain.ainvoke({'chat_history': history, 'input': question})
    needs_rag = parse_decision(decision)
    print(f'  [Node 1] {question!r} → {decision} (needs_rag={needs_rag})')
    return {'needs_rag': needs_rag, 'context': ''}                          # 흐름도 : 상태 갱신 needs_rag, context=""

async def retriever_node(state: AgentState) -> dict:                        # [Node 2]
    history, question = state.messages[:-1], state.messages[-1].content
    query = await rephrase_chain.ainvoke({'chat_history': history, 'input': question}) if history else question
    found = await retriever.ainvoke(query)
    print(f"  [Node 2] 재구성 : {query} → {[d.metadata['faq_id'] for d in found]}")
    return {'context': '\n\n'.join(d.page_content for d in found)}

async def answer_node(state: AgentState) -> dict:                           # [Node 3] 하나의 노드에서 두 갈래
    history, question = state.messages[:-1], state.messages[-1].content
    if state.needs_rag:
        response = await rag_qa_chain.ainvoke({'context': state.context, 'chat_history': history, 'input': question})
    else:
        response = await general_qa_chain.ainvoke({'chat_history': history, 'input': question})
    print(f"  [Node 3] {'RAG' if state.needs_rag else 'NO_RAG'} 답변 : {response.content[:55]}")
    return {'messages': [response]}

def route_by_rag_need(state: AgentState) -> Literal['retriever_node', 'answer_node']:
    return 'retriever_node' if state.needs_rag else 'answer_node'

workflow = StateGraph(AgentState)
workflow.add_node('classifier_node', classifier_node)
workflow.add_node('retriever_node', retriever_node)
workflow.add_node('answer_node', answer_node)
workflow.set_entry_point('classifier_node')
workflow.add_conditional_edges('classifier_node', route_by_rag_need,
                               {'retriever_node': 'retriever_node', 'answer_node': 'answer_node'})
workflow.add_edge('retriever_node', 'answer_node')
workflow.add_edge('answer_node', END)
print(workflow.compile().get_graph().draw_mermaid().split('graph TD;')[1].split('classDef')[0])


	__start__([<p>__start__</p>]):::first
	classifier_node(classifier_node)
	retriever_node(retriever_node)
	answer_node(answer_node)
	__end__([<p>__end__</p>]):::last
	__start__ --> classifier_node;
	classifier_node -.-> answer_node;
	classifier_node -.-> retriever_node;
	retriever_node --> answer_node;
	answer_node --> __end__;
	


- 흐름도에는 `[Node 3] answer_node` 상자가 **두 번** 그려져 있다(왼쪽 아래 상자 → 오른쪽 상자). 실제로는 노드 이름이 같으면 **하나의 노드**다 — 위 Mermaid 처럼 `retriever_node → answer_node` 와 `classifier_node -.-> answer_node` 두 길이 같은 상자로 들어온다
- 📝 `answer_node` 가 `state.needs_rag` 로 체인을 고른다 — **분기 판단이 두 곳**(라우터 · 답변 노드)에 있다. Day 055 정리판처럼 `rag_answer_node` · `general_answer_node` 로 나누면 그래프 그림만으로 어떤 답변을 했는지 보인다. 노드 수는 늘지만 한 노드가 한 가지 일만 한다(§4 비교)

## 2. 흐름도의 루프 실행 — `exit` 까지

In [8]:
# 🧪 LLM 대신 '정해 둔 규칙으로 답하는' 가짜 LLM — 체크포인터가 무엇을 저장 · 복원하는지 확인용
# 🧪 OpenAI 임베딩 대신 로컬 한국어 임베딩(jhgan/ko-sroberta-multitask)으로 실제 실행 — 순위 · 거리는 text-embedding-3-small 과 다를 수 있다

db_file = 'flowchart_agent.sqlite'
if os.path.exists(db_file):
    os.remove(db_file)

async def main(inputs):
    async with AsyncSqliteSaver.from_conn_string(db_file) as memory:           # SQLite 연결
        agent = workflow.compile(checkpointer=memory)
        config = {'configurable': {'thread_id': 'user-8888'}}
        for user_input in inputs:                                            # input() 대신 입력 목록
            print(f'사용자: {user_input}')
            if user_input.strip().lower() == 'exit':                         # exit 입력? → 예
                print('챗봇 종료 및 SQLite 연결 해제'); break
            before = len([h async for h in agent.aget_state_history(config)])
            final_state = await agent.ainvoke({'messages': [HumanMessage(user_input)]}, config=config)
            after = len([h async for h in agent.aget_state_history(config)])
            print(f"  → 최신 AIMessage 출력 · messages {len(final_state['messages'])}개 · 이번 턴 체크포인트 {after - before}개\n")
    return agent

agent = await main(['안녕하세요!', '수강 신청한 강의를 환불하고 싶어요', '그거 전액 다 받으려면 조건이 어떻게 되나요?', '고마워요', 'exit'])

사용자: 안녕하세요!
  [Node 1] '안녕하세요!' → NO_RAG (needs_rag=False)
  [Node 3] NO_RAG 답변 : (모의 일반 답변 · 기록 0개) 네, 무엇이든 물어보세요!
  → 최신 AIMessage 출력 · messages 2개 · 이번 턴 체크포인트 4개

사용자: 수강 신청한 강의를 환불하고 싶어요
  [Node 1] '수강 신청한 강의를 환불하고 싶어요' → RAG (needs_rag=True)
  [Node 2] 재구성 : 안녕하세요! 수강 신청한 강의를 환불하고 싶어요 → ['FAQ-002']
  [Node 3] RAG 답변 : (모의 · 기록 2개) 네, 환불 가능합니다.
  → 최신 AIMessage 출력 · messages 4개 · 이번 턴 체크포인트 5개

사용자: 그거 전액 다 받으려면 조건이 어떻게 되나요?
  [Node 1] '그거 전액 다 받으려면 조건이 어떻게 되나요?' → RAG (needs_rag=True)
  [Node 2] 재구성 : 수강 신청한 강의를 환불하고 싶어요 그거 전액 다 받으려면 조건이 어떻게 되나요? → ['FAQ-002']
  [Node 3] RAG 답변 : (모의 · 기록 4개) 네, 환불 가능합니다.
  → 최신 AIMessage 출력 · messages 6개 · 이번 턴 체크포인트 5개

사용자: 고마워요
  [Node 1] '고마워요' → NO_RAG (needs_rag=False)
  [Node 3] NO_RAG 답변 : (모의 일반 답변 · 기록 6개) 네, 무엇이든 물어보세요!


  → 최신 AIMessage 출력 · messages 8개 · 이번 턴 체크포인트 4개

사용자: exit
챗봇 종료 및 SQLite 연결 해제


| 턴 | 경로 | 이번 턴 체크포인트 |
|---|---|---|
| 안녕하세요! · 고마워요 | `classifier → answer` | 4개 (입력 · 시작 · 노드 2개) |
| 환불 · 그거 전액 | `classifier → retriever → answer` | 5개 (입력 · 시작 · 노드 3개) |

- 📝 흐름도의 '**3. 상태 저장** (최종 AgentState 영속화)' 은 끝에 한 번 저장하는 것처럼 보이지만, 체크포인터는 **노드가 끝날 때마다** 저장한다(위 표 · Day 055). 그래서 중간 노드에서 오류가 나도 그 앞까지는 남는다
- 흐름도 '1. 상태 복원' 의 '`add_messages` 리듀서: 과거 대화 + 새 HumanMessage 병합' 은 정확하다 — 턴마다 messages 가 2개씩 늘었다

In [9]:
# 흐름도의 'SQLite 연결 해제' 뒤 — 같은 agent 를 쓰면?
try:
    await agent.ainvoke({'messages': [HumanMessage('또 질문')]}, config={'configurable': {'thread_id': 'user-8888'}})
except ValueError as e:
    print('ValueError :', e)

# 다시 연결하면 대화가 이어진다
async with AsyncSqliteSaver.from_conn_string(db_file) as memory:
    agent = workflow.compile(checkpointer=memory)
    snap = await agent.aget_state({'configurable': {'thread_id': 'user-8888'}})
    print('다시 연결 — 저장된 messages :', len(snap.values['messages']), '개 · needs_rag :', snap.values['needs_rag'], '· context :', repr(snap.values['context']))

ValueError : no active connection
다시 연결 — 저장된 messages : 8 개 · needs_rag : False · context : ''


- 마지막 턴('고마워요', NO_RAG)이 끝난 상태의 `context` 는 `''` — 분류 노드가 매 턴 비운 덕분이다(흐름도 Node 1 의 `context=""`). Day 055 에서 이것이 없으면 앞 턴 매뉴얼이 남는 것을 확인했다

In [ ]:
# ⏳ API 키가 필요한 셀 — 정리 환경에서는 실행하지 않음 (직접 실행해 보기)

# 필기 main() 루프 그대로 — 직접 입력
async def chat():
    async with AsyncSqliteSaver.from_conn_string('multi_agent.sqlite') as memory:
        agent = workflow.compile(checkpointer=memory)
        config = {'configurable': {'thread_id': 'user-8888'}}
        while True:
            user_input = input('사용자: ')
            if user_input.strip().lower() == 'exit':
                print('챗봇을 안전하게 종료합니다.'); break
            final_state = await agent.ainvoke({'messages': [HumanMessage(content=user_input)]}, config=config)
            print(f"AI 에이전트:\n{final_state['messages'][-1].content}\n" + '-' * 80)

await chat()

## 3. 대화가 길어지면 — 프롬프트에 들어가는 기록

In [11]:
# 🧪 LLM 대신 '정해 둔 규칙으로 답하는' 가짜 LLM — 체크포인터가 무엇을 저장 · 복원하는지 확인용

async with AsyncSqliteSaver.from_conn_string(db_file) as memory:
    agent = workflow.compile(checkpointer=memory)
    cfg = {'configurable': {'thread_id': 'user-long'}}
    sizes = []
    for i in range(1, 7):
        state = await agent.ainvoke({'messages': [HumanMessage(f'{i}번째 질문 : 환불 규정 다시 알려줘')]}, cfg)
        history = state['messages']                      # 다음 턴의 chat_history = 지금까지의 모든 메시지
        sizes.append((i, len(history), sum(len(m.content) for m in history)))
for turn, n, chars in sizes:
    print(f'{turn}턴 끝 — 다음 턴 chat_history {n}개 · {chars}자')

  [Node 1] '1번째 질문 : 환불 규정 다시 알려줘' → RAG (needs_rag=True)
  [Node 2] 재구성 : 1번째 질문 : 환불 규정 다시 알려줘 → ['FAQ-002']
  [Node 3] RAG 답변 : (모의 · 기록 0개) 네, 환불 가능합니다.
  [Node 1] '2번째 질문 : 환불 규정 다시 알려줘' → RAG (needs_rag=True)
  [Node 2] 재구성 : 1번째 질문 : 환불 규정 다시 알려줘 2번째 질문 : 환불 규정 다시 알려줘 → ['FAQ-002']
  [Node 3] RAG 답변 : (모의 · 기록 2개) 네, 환불 가능합니다.
  [Node 1] '3번째 질문 : 환불 규정 다시 알려줘' → RAG (needs_rag=True)


  [Node 2] 재구성 : 2번째 질문 : 환불 규정 다시 알려줘 3번째 질문 : 환불 규정 다시 알려줘 → ['FAQ-002']
  [Node 3] RAG 답변 : (모의 · 기록 4개) 네, 환불 가능합니다.
  [Node 1] '4번째 질문 : 환불 규정 다시 알려줘' → RAG (needs_rag=True)
  [Node 2] 재구성 : 3번째 질문 : 환불 규정 다시 알려줘 4번째 질문 : 환불 규정 다시 알려줘 → ['FAQ-002']
  [Node 3] RAG 답변 : (모의 · 기록 6개) 네, 환불 가능합니다.
  [Node 1] '5번째 질문 : 환불 규정 다시 알려줘' → RAG (needs_rag=True)
  [Node 2] 재구성 : 4번째 질문 : 환불 규정 다시 알려줘 5번째 질문 : 환불 규정 다시 알려줘 → ['FAQ-002']
  [Node 3] RAG 답변 : (모의 · 기록 8개) 네, 환불 가능합니다.
  [Node 1] '6번째 질문 : 환불 규정 다시 알려줘' → RAG (needs_rag=True)


  [Node 2] 재구성 : 5번째 질문 : 환불 규정 다시 알려줘 6번째 질문 : 환불 규정 다시 알려줘 → ['FAQ-002']
  [Node 3] RAG 답변 : (모의 · 기록 10개) 네, 환불 가능합니다.
1턴 끝 — 다음 턴 chat_history 2개 · 46자
2턴 끝 — 다음 턴 chat_history 4개 · 92자
3턴 끝 — 다음 턴 chat_history 6개 · 138자
4턴 끝 — 다음 턴 chat_history 8개 · 184자
5턴 끝 — 다음 턴 chat_history 10개 · 230자
6턴 끝 — 다음 턴 chat_history 12개 · 277자


- 체크포인터가 기록을 **모두** 복원하므로 `chat_history` 는 턴마다 2개씩 늘고, 분류 · 재구성 · 답변 체인 **세 곳 모두**에 매번 들어간다 → 긴 대화에서는 토큰이 빠르게 늘어난다. 노드에서 `state.messages[:-1][-6:]` 처럼 최근 몇 턴만 넘기거나 `trim_messages` 를 쓰자 (DB 에는 전부 남는다)

## 4. 답변 노드 1개(흐름도) vs 2개(Day 055 정리판)

In [12]:
async def rag_answer_node(state: AgentState) -> dict:
    return {'messages': [await rag_qa_chain.ainvoke({'context': state.context, 'chat_history': state.messages[:-1], 'input': state.messages[-1].content})]}
async def general_answer_node(state: AgentState) -> dict:
    return {'messages': [await general_qa_chain.ainvoke({'chat_history': state.messages[:-1], 'input': state.messages[-1].content})]}

two = StateGraph(AgentState)
for n, f in [('classifier_node', classifier_node), ('retriever_node', retriever_node),
             ('rag_answer_node', rag_answer_node), ('general_answer_node', general_answer_node)]:
    two.add_node(n, f)
two.set_entry_point('classifier_node')
two.add_conditional_edges('classifier_node', lambda s: 'retriever_node' if s.needs_rag else 'general_answer_node',
                          {'retriever_node': 'retriever_node', 'general_answer_node': 'general_answer_node'})
two.add_edge('retriever_node', 'rag_answer_node'); two.add_edge('rag_answer_node', END); two.add_edge('general_answer_node', END)

for name, g in [('흐름도 (answer_node 1개)', workflow), ('Day 055 (답변 노드 2개)', two)]:
    edges = [l.strip() for l in g.compile().get_graph().draw_mermaid().splitlines() if ('--' in l or '-.' in l) and not l.startswith('---')]
    print(f'■ {name}'); [print('   ', e) for e in edges]

■ 흐름도 (answer_node 1개)
    __start__ --> classifier_node;
    classifier_node -.-> answer_node;
    classifier_node -.-> retriever_node;
    retriever_node --> answer_node;
    answer_node --> __end__;
■ Day 055 (답변 노드 2개)
    __start__ --> classifier_node;
    classifier_node -.-> general_answer_node;
    classifier_node -.-> retriever_node;
    retriever_node --> rag_answer_node;
    general_answer_node --> __end__;
    rag_answer_node --> __end__;


| | 흐름도 (`answer_node` 1개) | Day 055 정리판 (2개) |
|---|---|---|
| 노드 수 | 3 | 4 |
| 분기 판단 | 라우터 + `answer_node` 안의 `if` (두 곳) | 라우터 한 곳 |
| 그림 · `stream` 로 보이는 것 | 어느 답변 체인을 썼는지 안 보임 | 노드 이름으로 보임 |
| 장점 | 그래프가 단순 · 답변 후처리(로그 · 형식)를 한 곳에 | 한 노드 = 한 일 · 따로 시험하기 쉬움 |

둘 다 같은 답을 낸다 — 취향 · 팀 규칙의 문제다. 다만 '분기 판단을 두 곳에 두면 한쪽만 고쳤을 때 어긋난다' 는 점은 기억해 두자.

## 정리

| 흐름도 단계 | 코드 |
|---|---|
| 입력 · exit | `async with AsyncSqliteSaver… as memory:` 안의 `while` — `exit` 면 `break` → 블록이 끝나며 연결 해제 |
| 1. 상태 복원 | `ainvoke({'messages': [HumanMessage]}, config={'configurable': {'thread_id': …}})` |
| 2. 워크플로우 | `classifier(context='')` → `route_by_rag_need` → `retriever` → `answer` / `answer` |
| 3. 상태 저장 | 노드가 끝날 때마다 자동 (한 턴 4~5개) |
| 출력 | `final_state['messages'][-1].content` |